# InternSpark Data Science Internship — Zomato Dataset Analysis

Analyze restaurant data for ratings, cuisines, locations, price ranges, and review text. Place the downloaded CSV file(s) inside `data/` before running the notebook.

**Outputs:** cleaned CSV, charts, findings, recommendations, and a PDF report in `outputs/`.

## 1. Install/import libraries
Run the install command in a VS Code terminal once: `pip install -r requirements.txt`

In [ ]:
from pathlib import Path
import re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, PageBreak
from reportlab.lib.styles import getSampleStyleSheet

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
OUT_DIR = ROOT / 'outputs'
OUT_DIR.mkdir(exist_ok=True)
sns.set_theme()
pd.set_option('display.max_columns', 100)


## 2. Load restaurant CSV
This searches `data/` for CSV files. If multiple are present, it combines files with matching column layouts.

In [ ]:
csv_files = list(DATA_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError('No CSV found. Download/extract the dataset and put its CSV file(s) in the data folder.')

frames = []
for file in csv_files:
    try:
        temp = pd.read_csv(file, encoding='utf-8', low_memory=False)
    except UnicodeDecodeError:
        temp = pd.read_csv(file, encoding='latin1', low_memory=False)
    temp.columns = [str(c).strip() for c in temp.columns]
    frames.append(temp)
df = pd.concat(frames, ignore_index=True, sort=False)
print('Files:', [f.name for f in csv_files])
print('Rows/columns:', df.shape)
display(df.head())
print(df.columns.tolist())


## 3. Clean the data
Standardize column names, remove duplicate rows, convert common numeric fields, and handle blank text.

In [ ]:
def clean_col_name(name):
    name = str(name).strip().lower()
    name = re.sub(r'[^a-z0-9]+', '_', name)
    return name.strip('_')

df.columns = [clean_col_name(c) for c in df.columns]
df = df.replace(r'^\s*$', np.nan, regex=True)
before = len(df)
df = df.drop_duplicates().copy()
print('Duplicate rows removed:', before - len(df))

numeric_candidates = ['aggregate_rating','rating','average_cost_for_two','price_range','votes','longitude','latitude','country_code']
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

text_cols = df.select_dtypes(include='object').columns
for col in text_cols:
    df[col] = df[col].astype('string').str.strip()

if 'aggregate_rating' in df.columns:
    df.loc[~df['aggregate_rating'].between(0, 5), 'aggregate_rating'] = np.nan
df.to_csv(OUT_DIR / 'zomato_cleaned.csv', index=False)
print('Cleaned shape:', df.shape)
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame('missing_values'))


## 4. Identify key columns
The dataset may use slightly different column names. This helper selects the closest available field.

In [ ]:
def find_col(*names):
    for name in names:
        if name in df.columns:
            return name
    return None

rating_col = find_col('aggregate_rating','rating')
cuisine_col = find_col('cuisines','cuisine')
location_col = find_col('locality','city','location')
price_col = find_col('average_cost_for_two','average_cost','cost_for_two')
price_range_col = find_col('price_range')
currency_col = find_col('currency')
review_col = find_col('review','reviews','review_text','review_texts','text')

print({'rating':rating_col,'cuisine':cuisine_col,'location':location_col,
       'cost':price_col,'price range':price_range_col,'currency':currency_col,
       'review text':review_col})


## 5. Explore rating distribution


In [ ]:
if rating_col:
    ratings = df[rating_col].dropna()
    plt.figure(figsize=(8,5))
    sns.histplot(ratings, bins=20, kde=True)
    plt.title('Restaurant Rating Distribution')
    plt.xlabel('Rating'); plt.ylabel('Number of restaurants')
    plt.tight_layout(); plt.savefig(OUT_DIR/'rating_distribution.png', dpi=160); plt.show()
    print('Mean rating:', round(ratings.mean(),2), '| Median:', round(ratings.median(),2))
else:
    print('Rating column not found; skipping rating distribution.')


## 6. Cuisine vs rating
Split multi-cuisine entries and compare average rating for cuisines with at least 20 restaurants (adjust threshold for smaller datasets).

In [ ]:
if cuisine_col and rating_col:
    cuisine_data = df[[cuisine_col, rating_col]].dropna().copy()
    cuisine_data[cuisine_col] = cuisine_data[cuisine_col].astype(str).str.split(',')
    cuisine_data = cuisine_data.explode(cuisine_col)
    cuisine_data[cuisine_col] = cuisine_data[cuisine_col].str.strip()
    cuisine_stats = cuisine_data.groupby(cuisine_col)[rating_col].agg(['mean','count']).query('count >= 20').sort_values('mean', ascending=False)
    display(cuisine_stats.head(15))
    if not cuisine_stats.empty:
        cuisine_stats.head(15).sort_values('mean').plot(kind='barh', y='mean', figsize=(9,6), legend=False)
        plt.title('Top cuisines by average rating (min 20 restaurants)'); plt.xlabel('Average rating')
        plt.tight_layout(); plt.savefig(OUT_DIR/'cuisine_vs_rating.png', dpi=160); plt.show()
else:
    print('Cuisine or rating column not found; skipping cuisine comparison.')


## 7. Location hotspots


In [ ]:
if location_col:
    top_locations = df[location_col].dropna().value_counts().head(15)
    display(top_locations.to_frame('restaurant_count'))
    top_locations.sort_values().plot(kind='barh', figsize=(9,6))
    plt.title('Restaurant Hotspots'); plt.xlabel('Restaurant count')
    plt.tight_layout(); plt.savefig(OUT_DIR/'location_hotspots.png', dpi=160); plt.show()
else:
    print('Location column not found; skipping location analysis.')


## 8. Price vs rating
If currency is present, compare costs within each currency instead of mixing currencies.

In [ ]:
if price_col and rating_col:
    plot_cols = [price_col, rating_col] + ([currency_col] if currency_col else [])
    price_data = df[plot_cols].dropna(subset=[price_col, rating_col]).copy()
    if currency_col:
        common_currency = price_data[currency_col].value_counts().index[0]
        price_data = price_data[price_data[currency_col] == common_currency]
        print('Showing price analysis for currency:', common_currency)
    price_data = price_data[(price_data[price_col] >= 0) & (price_data[price_col] <= price_data[price_col].quantile(.99))]
    plt.figure(figsize=(8,5))
    sns.regplot(data=price_data, x=price_col, y=rating_col, scatter_kws={'alpha':.25}, line_kws={'color':'red'})
    plt.title('Cost vs Rating (most common currency; top 1% cost trimmed)')
    plt.tight_layout(); plt.savefig(OUT_DIR/'price_vs_rating.png', dpi=160); plt.show()
else:
    print('Cost or rating column not found; skipping price analysis.')


## 9. Heatmap of numeric relationships


In [ ]:
numeric_df = df.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    corr = numeric_df.corr(numeric_only=True)
    plt.figure(figsize=(10,7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Numeric Feature Correlation Heatmap')
    plt.tight_layout(); plt.savefig(OUT_DIR/'correlation_heatmap.png', dpi=160); plt.show()
else:
    print('Not enough numeric columns for a correlation heatmap.')


## 10. Wordcloud: review text or popular cuisines
Some restaurant datasets contain no review text. If so, the notebook uses cuisine names as a fallback wordcloud.

In [ ]:
if review_col and df[review_col].notna().any():
    words = ' '.join(df[review_col].dropna().astype(str).tolist())
    wordcloud_title = 'Common words in reviews'
elif cuisine_col and df[cuisine_col].notna().any():
    words = ' '.join(df[cuisine_col].dropna().astype(str).tolist())
    wordcloud_title = 'Popular cuisine terms (review text not available)'
else:
    words = ''
    wordcloud_title = 'No text field available'

if words.strip():
    wc = WordCloud(width=1200, height=600, background_color='white', stopwords=STOPWORDS, collocations=False).generate(words)
    plt.figure(figsize=(12,6)); plt.imshow(wc, interpolation='bilinear'); plt.axis('off')
    plt.title(wordcloud_title); plt.tight_layout(); plt.savefig(OUT_DIR/'wordcloud.png', dpi=160); plt.show()
else:
    print('No usable review or cuisine text for wordcloud.')


## 11. Findings and 5 recommendations
Recommendations are general platform ideas; adjust them to match the actual charts and findings from your dataset.

In [ ]:
findings = []
if rating_col and df[rating_col].notna().any():
    findings.append(f"Average restaurant rating: {df[rating_col].mean():.2f}/5.")
if location_col and df[location_col].notna().any():
    loc = df[location_col].value_counts().index[0]
    findings.append(f"Largest restaurant listing hotspot: {loc}.")
if cuisine_col and df[cuisine_col].notna().any():
    findings.append(f"Most frequently listed cuisine entry: {df[cuisine_col].value_counts().index[0]}.")
if price_range_col and df[price_range_col].notna().any():
    findings.append(f"Most common price range: {df[price_range_col].mode().iloc[0]}.")

recommendations = [
    'Partnerships: onboard more highly rated restaurants in under-served localities, not only the largest hotspots.',
    'Discovery: create cuisine-based collections using popular cuisines and highly rated restaurant listings.',
    'Trust: highlight verified rating and review counts together so users can judge rating reliability.',
    'Value: build price-range filters and value-for-money collections; compare costs only within the same currency.',
    'Content: publish locality and cuisine guides, seasonal food lists, and short restaurant feature posts based on dataset trends.'
]
print('KEY FINDINGS')
for item in findings: print('-', item)
print('\n5 PLATFORM RECOMMENDATIONS')
for i, item in enumerate(recommendations, 1): print(f'{i}. {item}')


## 12. Generate PDF report
This report includes findings, recommendations, and the charts generated above.

In [ ]:
styles = getSampleStyleSheet()
story = [Paragraph('Zomato Dataset Analysis', styles['Title']),
         Paragraph('InternSpark Data Science Internship Project', styles['Heading2']),
         Spacer(1, 12), Paragraph('Key Findings', styles['Heading1'])]
for item in findings:
    story.append(Paragraph('• ' + item, styles['BodyText'])); story.append(Spacer(1, 5))
story.append(Paragraph('Recommendations', styles['Heading1']))
for i, item in enumerate(recommendations, 1):
    story.append(Paragraph(f'{i}. {item}', styles['BodyText'])); story.append(Spacer(1, 5))
chart_files = ['rating_distribution.png','cuisine_vs_rating.png','location_hotspots.png','price_vs_rating.png','correlation_heatmap.png','wordcloud.png']
for chart in chart_files:
    path = OUT_DIR / chart
    if path.exists():
        story.append(PageBreak()); story.append(Paragraph(chart.replace('_',' ').replace('.png','').title(), styles['Heading1']))
        story.append(RLImage(str(path), width=470, height=250))
pdf_path = OUT_DIR / 'zomato_analysis_report.pdf'
SimpleDocTemplate(str(pdf_path), pagesize=A4, title='Zomato Dataset Analysis').build(story)
print('PDF saved to:', pdf_path.resolve())


## Project completion checklist
- [ ] CSV file(s) placed in `data/`
- [ ] All notebook cells run from top to bottom
- [ ] Check generated charts and findings against the actual dataset
- [ ] Submit the notebook and `outputs/zomato_analysis_report.pdf`
- [ ] Push the project to GitHub